# Notebook 01b5 — Static Spatial Features

## README

### Overview

This notebook enriches the **1 km grid** produced by `00a` with **static, time-invariant
spatial features** per `spatial_id`: population, building height, road length by type,
public transport counts, point-of-interest counts by category, and fossil-fuel power
plant counts.

It sits **between `01b_raster_to_grid_aggregation.ipynb` and `01c_feature_engineering.ipynb`**,
because:
- it needs the **grid** to exist (produced by `00a`, the same grid used by `01b`)
- it has **nothing to do with the daily NO₂/temperature panel** that `01b` builds, or with
  the TCI merge / lag features that `01c` performs
- its output is a **static per-tile table** (no `date` column) that `01c` can broadcast
  across all dates when it merges TCI

### Why this differs from the original notebook it is adapted from

The notebook this was adapted from processed **one city at a time** with hardcoded
`01_data/` / `02_outputs/` paths and separate per-city POI/road files supplied manually.
This version is **city-agnostic** (`CITIES` list, one `USER INPUTS` cell) and resolves
all inputs from files already produced earlier in this pipeline:

| Feature | Source notebook | Path |
|---|---|---|
| Grid | `00a_city_boundary_grid_generation.ipynb` | `data/processed/{city}/grid/grid_{city}_1km.gpkg` |
| Road network | `00e_poi_road_network_download.ipynb` | `data/raw/{city}/road_network/{city}_roads_edges.gpkg` |
| Public transport platforms | `00e_poi_road_network_download.ipynb` | `data/raw/{city}/public_transport/{city}_pt_platforms.gpkg` |
| POI: commercial / offices / education / leisure | `00e_poi_road_network_download.ipynb` | `data/raw/{city}/poi/{city}_poi_{category}.gpkg` |
| Building height | `00d_gee_download.ipynb` (GHSL export) | `data/raw/{city}/building_height/{city}_building_height.tif` |
| Population | **user-supplied global raster** | `data/aux/population/{POP_RASTER_FILENAME}` |
| Power plants | **user-supplied global GeoPackage** | `data/aux/power_plants/{POWER_PLANT_FILENAME}` |

Population and power plants remain user-supplied local files (consistent with the
Natural Earth land/lake convention already used in `00a`) since they are single
global datasets, not per-city satellite downloads.

### Inputs (per city)

| Source | Path | Notes |
|---|---|---|
| `00a` | `data/processed/{city}/grid/grid_{city}_1km.gpkg` | Must contain `spatial_id` |
| `00d` | `data/raw/{city}/building_height/{city}_building_height.tif` | GHSL, optional |
| `00e` | `data/raw/{city}/road_network/{city}_roads_edges.gpkg` | `highway` column required |
| `00e` | `data/raw/{city}/public_transport/{city}_pt_platforms.gpkg` | optional |
| `00e` | `data/raw/{city}/poi/{city}_poi_{commercial,offices,education,leisure}.gpkg` | optional, per category |

### Global inputs (user-supplied, once)

| Variable | Path |
|---|---|
| World population raster | `data/aux/population/{POP_RASTER_FILENAME}` |
| Global power plant database | `data/aux/power_plants/{POWER_PLANT_FILENAME}` |

### Output (per city)

```
data/processed/{city}/grid_panel/
  {city}_static_features.gpkg    ← spatial_id + all static features + geometry
  {city}_static_features.csv     ← same, without geometry (lightweight join table)
```

### Output schema

```
spatial_id | tile_pop | building_height |
road_length_<highway_type>_m ... | road_length_total_m |
n_bus_stops | n_mass_transit_lines | n_mass_transit_polygons | n_public_transport_total |
n_commercial | n_offices | n_education | n_leisure | n_power_plant
```

All count and length columns are filled with `0` where no features were found in a
tile (never `NaN`) — a tile with zero roads or POIs genuinely has zero, not missing data.
`tile_pop` and `building_height` are left as `NaN` where the source raster has no
coverage, since zero population/height would be a false claim about a tile that simply
wasn't sampled.

### Pipeline position

```
00a_city_boundary_grid_generation.ipynb
00e_poi_road_network_download.ipynb
00d_gee_download.ipynb (building height)
        │
        ▼
01a_pixel_filling.ipynb
        │
        ▼
01b_raster_to_grid_aggregation.ipynb
        │
        ▼
01b5_spatial_features.ipynb   ◄── YOU ARE HERE
        │
        ▼
01c_feature_engineering.ipynb   (merges static features + TCI + lag)
        │
        ▼
01d_monthly_tile_panel.ipynb
```

### Design principles (consistent with the rest of the pipeline)

- **City-agnostic**: add cities by appending to `CITIES`; everything else is generic.
- **USER INPUTS cell**: all paths and toggles in one place at the top.
- **Graceful degradation**: each feature block is wrapped so that a missing optional
  input (e.g. no leisure POIs downloaded yet) produces an all-zero column with a
  printed warning, rather than crashing the whole city.
- **FAST_DEV_MODE**: not applicable here (no per-date loop — this is a one-shot
  per-city spatial join), but a `SKIP_EXISTING` flag gives the same resume-safety
  the rest of the pipeline uses.
- **City-specific UTM CRS**: road length and spatial joins use the same `CITY_CRS`
  mapping from `src/config.py` used throughout the pipeline, not Web Mercator.
- **Visual QC kept inline**: one heatmap per feature block, matplotlib `Agg`-safe,
  for visual inspection — consistent with `01a`/`01d` plotting conventions.


## 0 · Initialisation

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import fiona
import matplotlib
matplotlib.use("Agg")   # headless-safe; figures are still shown via plt.show() in Jupyter
import matplotlib.pyplot as plt

try:
    import contextily as cx
    HAS_CONTEXTILY = True
except ImportError:
    HAS_CONTEXTILY = False
    print("contextily not installed — basemap layers will be skipped in plots.")

from exactextract import exact_extract

warnings.filterwarnings("ignore")

# ── Project root & src path (walks up from cwd until src/ is found) ───────────
def find_project_root(start: Path, marker: str = "src") -> Path:
    for parent in [start, *start.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find '{marker}/' above {start}")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from config import DATA_PATH, CITY_CRS
    print("Loaded DATA_PATH and CITY_CRS from src/config.py")
except ImportError:
    print("WARNING: src/config.py not found — DATA_PATH/CITY_CRS must be set manually.")
    DATA_PATH = PROJECT_ROOT / "data"
    CITY_CRS = {}

RAW_DIR       = DATA_PATH / "raw"
PROCESSED_DIR = DATA_PATH / "processed"
AUX_DIR       = DATA_PATH / "aux"

print(f"Project root  : {PROJECT_ROOT}")
print(f"Data root     : {DATA_PATH}")

## 1 · USER INPUTS

Edit only this cell to configure cities, global auxiliary data paths, and POI
categories. Everything downstream resolves per-city paths automatically from
the conventions used by `00a`, `00d`, and `00e`.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ── USER INPUTS  (edit this cell to configure the run) ───────────────────────
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ── Cities to process ─────────────────────────────────────────────────────────
# Must match the city slugs used in 00a/00d/00e (data/raw/{city}/, data/processed/{city}/).
CITIES = [
    "abuja",
    "algiers",
    "baghdad",
    "buenos_aires",
    "cairo",
    "cape_town",
    "dakar",
    "lima",
    "los_angeles",
    "madrid",
    "mexico_city",
    "mumbai",
    "new_york",
    "santiago",
    "yangon",
]

# ── POI categories ─────────────────────────────────────────────────────────────
# Must match the standalone GeoPackages downloaded by 00e:
#   data/raw/{city}/poi/{city}_poi_{category}.gpkg
POI_CATEGORIES = ["poi_commercial", "poi_offices", "poi_education", "poi_leisure"]

# ── Global auxiliary inputs (user-supplied, placed once under data/aux/) ──────
# Consistent with the Natural Earth land/lake convention used in 00a.
POP_RASTER_PATH    = AUX_DIR / "population"   / "world_pop.tif"
POWER_PLANT_PATH   = AUX_DIR / "power_plants" / "global_power_plant_database.gpkg"

# Power plant fuel types treated as "fossil fuel" for the n_power_plant count.
FOSSIL_FUEL_TYPES = ["coal", "oil", "gas"]

# ── Run options ─────────────────────────────────────────────────────────────────
SKIP_EXISTING = True   # skip a city entirely if its output GeoPackage already exists
MAKE_PLOTS    = True   # show one inline heatmap per feature block per city

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"Cities          : {len(CITIES)}")
print(f"POI categories  : {POI_CATEGORIES}")
print(f"Population raster : {POP_RASTER_PATH}  (exists: {POP_RASTER_PATH.exists()})")
print(f"Power plant DB    : {POWER_PLANT_PATH}  (exists: {POWER_PLANT_PATH.exists()})")

## 2 · Path Helpers

Resolves every per-city input path from the conventions used by `00a` (grid),
`00d` (building height), and `00e` (roads, public transport, POI categories).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PATH HELPERS  (do not edit — paths follow the conventions of 00a/00d/00e)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def grid_path(city: str) -> Path:
    """1 km grid produced by 00a."""
    return PROCESSED_DIR / city / "grid" / f"grid_{city}_1km.gpkg"

def building_height_path(city: str) -> Path:
    """GHSL building height raster produced by 00d (export_building_height)."""
    return RAW_DIR / city / "building_height" / f"{city}_building_height.tif"

def roads_edges_path(city: str) -> Path:
    """Road network edges produced by 00e."""
    return RAW_DIR / city / "road_network" / f"{city}_roads_edges.gpkg"

def pt_platforms_path(city: str) -> Path:
    """Public transport platforms produced by 00e."""
    return RAW_DIR / city / "public_transport" / f"{city}_pt_platforms.gpkg"

def poi_category_path(city: str, category: str) -> Path:
    """Standalone POI category GeoPackage produced by 00e, e.g. poi_commercial."""
    return RAW_DIR / city / "poi" / f"{city}_{category}.gpkg"

def output_dir(city: str) -> Path:
    p = PROCESSED_DIR / city / "grid_panel"
    p.mkdir(parents=True, exist_ok=True)
    return p

def output_gpkg_path(city: str) -> Path:
    return output_dir(city) / f"{city}_static_features.gpkg"

def output_csv_path(city: str) -> Path:
    return output_dir(city) / f"{city}_static_features.csv"


def utm_crs_for(city: str, fallback_gdf: gpd.GeoDataFrame = None) -> str:
    """
    Resolve the city's UTM CRS from CITY_CRS (src/config.py), falling back to
    GeoPandas' estimate_utm_crs() on the supplied GeoDataFrame if the city is
    not found there. Mirrors the CRS handling already used in 00a/01b.
    """
    if city in CITY_CRS:
        return CITY_CRS[city]
    if fallback_gdf is not None:
        print(f"  WARNING: '{city}' not found in CITY_CRS — estimating UTM zone instead.")
        return fallback_gdf.estimate_utm_crs()
    raise KeyError(f"City '{city}' not found in CITY_CRS and no fallback GeoDataFrame given.")


print("Path helpers defined.")

## 3 · Pre-flight Checks

Reports which inputs are present per city **before** processing. Missing optional
inputs (roads, POIs, public transport, building height) print a warning but do
not block the run — the corresponding feature block falls back to all-zero
columns for that city. A missing **grid** is the only hard blocker, since every
feature block needs it.

In [ ]:
print("=" * 70)
print("PRE-FLIGHT CHECKS")
print("=" * 70)

hard_blockers = []

for city in CITIES:
    print(f"\n📍 {city}")

    g_path = grid_path(city)
    g_ok   = g_path.exists()
    print(f"  Grid                [{'✓' if g_ok else '✗ MISSING — hard blocker'}]: {g_path}")
    if not g_ok:
        hard_blockers.append(city)

    bh_ok = building_height_path(city).exists()
    print(f"  Building height     [{'✓' if bh_ok else '— optional, not found'}]: {building_height_path(city)}")

    roads_ok = roads_edges_path(city).exists()
    print(f"  Road network        [{'✓' if roads_ok else '— optional, not found'}]: {roads_edges_path(city)}")

    pt_ok = pt_platforms_path(city).exists()
    print(f"  PT platforms        [{'✓' if pt_ok else '— optional, not found'}]: {pt_platforms_path(city)}")

    for cat in POI_CATEGORIES:
        p = poi_category_path(city, cat)
        print(f"  {cat:<19} [{'✓' if p.exists() else '— optional, not found'}]: {p}")

print("\n" + "=" * 70)
print(f"Global population raster : {'✓' if POP_RASTER_PATH.exists() else '✗ MISSING'}  ({POP_RASTER_PATH})")
print(f"Global power plant DB    : {'✓' if POWER_PLANT_PATH.exists() else '✗ MISSING'}  ({POWER_PLANT_PATH})")

if hard_blockers:
    raise FileNotFoundError(
        f"\nGrid missing for: {hard_blockers}\n"
        f"Run notebook 00a for these cities before proceeding."
    )
print("\n✅ All hard blockers cleared. Optional inputs missing above will yield")
print("   all-zero / NaN columns for the affected city, with a warning at run time.")

## 4 · Feature Functions

Each function takes a city slug and the loaded grid, returns a 2-column
DataFrame (`spatial_id` + the new feature), and optionally shows an inline
heatmap. All functions degrade gracefully if their source file is missing.

### 4.1 Population

In [ ]:
def compute_population(city: str, grid: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Sum world population raster values per tile using exactextract.
    Returns NaN (not 0) for tiles with no raster coverage, since a tile that
    was never sampled is not the same as a tile with zero people.
    """
    if not POP_RASTER_PATH.exists():
        print(f"  [{city}] Population raster not found — tile_pop will be NaN.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"], "tile_pop": np.nan})

    with rasterio.open(POP_RASTER_PATH) as src:
        grid_for_extract = grid.to_crs(src.crs) if grid.crs != src.crs else grid
        result = exact_extract(
            str(POP_RASTER_PATH),
            grid_for_extract,
            ["sum"],
            include_cols=["spatial_id"],
            output="pandas",
        )

    result = result.rename(columns={"sum": "tile_pop"})[["spatial_id", "tile_pop"]]

    if MAKE_PLOTS:
        plotted = grid.merge(result, on="spatial_id", how="left")
        fig, ax = plt.subplots(figsize=(8, 8))
        plotted.plot(column="tile_pop", ax=ax, legend=True, edgecolor="black",
                     linewidth=0.1, cmap="OrRd",
                     missing_kwds={"color": "lightgrey", "label": "No data"})
        if HAS_CONTEXTILY:
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=grid.crs)
            except Exception:
                pass
        ax.set_title(f"{city} — Population by tile")
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    print(f"  [{city}] Population — total across tiles: {result['tile_pop'].sum():,.0f}")
    return result


print("compute_population() defined.")

### 4.2 Building Height

In [ ]:
def compute_building_height(city: str, grid: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Mean building height per tile from the GHSL raster exported by 00d.
    Returns NaN for tiles with no raster coverage.
    """
    bh_path = building_height_path(city)
    if not bh_path.exists():
        print(f"  [{city}] Building height raster not found — building_height will be NaN.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"], "building_height": np.nan})

    with rasterio.open(bh_path) as src:
        grid_for_extract = grid.to_crs(src.crs) if grid.crs != src.crs else grid
        result = exact_extract(
            str(bh_path),
            grid_for_extract,
            ["mean"],
            include_cols=["spatial_id"],
            output="pandas",
        )

    result = result.rename(columns={"mean": "building_height"})[["spatial_id", "building_height"]]

    if MAKE_PLOTS:
        plotted = grid.merge(result, on="spatial_id", how="left")
        fig, ax = plt.subplots(figsize=(8, 8))
        plotted.plot(column="building_height", ax=ax, legend=True, edgecolor="black",
                     linewidth=0.1, cmap="OrRd",
                     missing_kwds={"color": "lightgrey", "label": "No data"})
        if HAS_CONTEXTILY:
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=grid.crs)
            except Exception:
                pass
        ax.set_title(f"{city} — Mean building height by tile")
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    return result


print("compute_building_height() defined.")

### 4.3 Road Length by Type

In [ ]:
def compute_road_length(city: str, grid: gpd.GeoDataFrame, projected_crs: str) -> pd.DataFrame:
    """
    Intersect the road network (00e) with the grid and sum road length per
    tile, split by OSM `highway` type, plus a road_length_total_m column.
    Returns all-zero columns (not NaN) if no road file is found, since a
    tile with no roads genuinely has zero road length.
    """
    roads_path = roads_edges_path(city)
    if not roads_path.exists():
        print(f"  [{city}] Road network not found — road_length_total_m will be 0.")
        return pd.DataFrame({
            "spatial_id": grid["spatial_id"],
            "road_length_total_m": 0.0,
        })

    roads = gpd.read_file(roads_path)
    if "highway" not in roads.columns:
        print(f"  [{city}] 'highway' column missing in roads file — road_length_total_m will be 0.")
        return pd.DataFrame({
            "spatial_id": grid["spatial_id"],
            "road_length_total_m": 0.0,
        })

    roads = roads[["highway", "geometry"]].dropna(subset=["highway", "geometry"])
    roads = roads[~roads.geometry.is_empty].copy()
    # highway can be a list-like string for multi-tagged ways — keep only the first value
    roads["highway"] = roads["highway"].apply(
        lambda v: v[0] if isinstance(v, (list, tuple)) else str(v).strip("[]'\" ").split(",")[0]
    )

    roads = roads.to_crs(grid.crs).to_crs(projected_crs)
    grid_proj = grid.to_crs(projected_crs)

    road_grid = gpd.overlay(roads, grid_proj[["spatial_id", "geometry"]], how="intersection")
    road_grid["road_length_m"] = road_grid.geometry.length

    road_summary = (
        road_grid.groupby(["spatial_id", "highway"])["road_length_m"].sum().reset_index()
    )
    wide = road_summary.pivot_table(
        index="spatial_id", columns="highway", values="road_length_m", fill_value=0
    ).reset_index()
    wide.columns.name = None
    wide = wide.rename(columns={
        c: f"road_length_{c}_m" for c in wide.columns if c != "spatial_id"
    })

    length_cols = [c for c in wide.columns if c.startswith("road_length_")]
    wide["road_length_total_m"] = wide[length_cols].sum(axis=1)

    # Ensure every tile in the grid appears, with 0 (not NaN) where no roads intersected
    full = grid[["spatial_id"]].merge(wide, on="spatial_id", how="left")
    fill_cols = [c for c in full.columns if c.startswith("road_length_")]
    full[fill_cols] = full[fill_cols].fillna(0)

    if MAKE_PLOTS:
        plotted = grid.merge(full, on="spatial_id", how="left")
        fig, ax = plt.subplots(figsize=(8, 8))
        plotted.plot(column="road_length_total_m", ax=ax, legend=True,
                     edgecolor="black", linewidth=0.1, cmap="OrRd")
        if HAS_CONTEXTILY:
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=grid.crs)
            except Exception:
                pass
        ax.set_title(f"{city} — Total road length by tile")
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    print(f"  [{city}] Road length — total across tiles: {full['road_length_total_m'].sum():,.0f} m")
    return full


print("compute_road_length() defined.")

### 4.4 Public Transport Counts

Classifies features by geometry type: points → bus stops, lines → mass transit
line segments, polygons → mass transit polygons (converted to representative
points before the spatial join, so a polygon contributes to exactly one tile).

In [ ]:
def _classify_geometry_type(geom) -> str:
    if geom.geom_type in ["Point", "MultiPoint"]:
        return "bus_stop"
    elif geom.geom_type in ["LineString", "MultiLineString"]:
        return "mass_transit_line"
    elif geom.geom_type in ["Polygon", "MultiPolygon"]:
        return "mass_transit_polygon"
    return "other"


def compute_public_transport(city: str, grid: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Count public transport platforms per tile by geometry type
    (00e: {city}_pt_platforms.gpkg). All-zero columns if the file is
    missing, empty, or has no usable layers.
    """
    count_cols = ["n_bus_stops", "n_mass_transit_lines",
                  "n_mass_transit_polygons", "n_public_transport_total"]

    pt_path = pt_platforms_path(city)
    if not pt_path.exists():
        print(f"  [{city}] PT platforms file not found — public transport counts will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"],
                              **{c: 0 for c in count_cols}})

    try:
        layers = fiona.listlayers(pt_path)
    except Exception as e:
        print(f"  [{city}] Could not read layers from {pt_path}: {e} — counts will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"],
                              **{c: 0 for c in count_cols}})

    frames = []
    for layer in layers:
        gdf = gpd.read_file(pt_path, layer=layer)
        if len(gdf) > 0:
            frames.append(gdf)

    if not frames:
        print(f"  [{city}] PT platforms file has no features — counts will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"],
                              **{c: 0 for c in count_cols}})

    pt = pd.concat(frames, ignore_index=True)
    pt = gpd.GeoDataFrame(pt, geometry="geometry", crs=frames[0].crs)
    pt = pt[~pt.geometry.is_empty].dropna(subset=["geometry"]).to_crs(grid.crs)
    pt["pt_type"] = pt.geometry.apply(_classify_geometry_type)

    def _count(subset: gpd.GeoDataFrame, predicate: str, out_name: str) -> pd.DataFrame:
        if len(subset) == 0:
            return pd.DataFrame(columns=["spatial_id", out_name])
        joined = gpd.sjoin(subset, grid[["spatial_id", "geometry"]],
                            how="inner", predicate=predicate)
        return joined.groupby("spatial_id").size().reset_index(name=out_name)

    pts  = pt[pt["pt_type"] == "bus_stop"].copy()
    lns  = pt[pt["pt_type"] == "mass_transit_line"].copy()
    polys = pt[pt["pt_type"] == "mass_transit_polygon"].copy()
    if len(polys) > 0:
        polys["geometry"] = polys.geometry.representative_point()

    point_counts   = _count(pts,  "within",     "n_bus_stops")
    line_counts    = _count(lns,  "intersects", "n_mass_transit_lines")
    polygon_counts = _count(polys, "within",    "n_mass_transit_polygons")

    out = grid[["spatial_id"]].copy()
    for df in [point_counts, line_counts, polygon_counts]:
        out = out.merge(df, on="spatial_id", how="left")

    base_cols = ["n_bus_stops", "n_mass_transit_lines", "n_mass_transit_polygons"]
    out[base_cols] = out[base_cols].fillna(0).astype(int)
    out["n_public_transport_total"] = out[base_cols].sum(axis=1)

    if MAKE_PLOTS:
        plotted = grid.merge(out, on="spatial_id", how="left")
        fig, ax = plt.subplots(figsize=(8, 8))
        plotted.plot(column="n_public_transport_total", ax=ax, legend=True,
                     edgecolor="black", linewidth=0.1, cmap="OrRd")
        if HAS_CONTEXTILY:
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=grid.crs)
            except Exception:
                pass
        ax.set_title(f"{city} — Public transport count by tile")
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    print(f"  [{city}] PT — bus stops: {out['n_bus_stops'].sum()}, "
          f"lines: {out['n_mass_transit_lines'].sum()}, "
          f"polygons: {out['n_mass_transit_polygons'].sum()}")
    return out[["spatial_id"] + count_cols]


print("compute_public_transport() defined.")

### 4.5 Point-of-Interest Counts

One function reused for every category in `POI_CATEGORIES` (commercial,
offices, education, leisure). Points are counted directly; polygons are
converted to representative points first so each contributes to exactly
one tile.

In [ ]:
def compute_poi_count(city: str, grid: gpd.GeoDataFrame, category: str) -> pd.DataFrame:
    """
    Count POIs of one category per tile (00e: {city}_{category}.gpkg).
    Output column is n_<short_name>, where short_name drops the 'poi_' prefix
    (e.g. category='poi_commercial' -> column 'n_commercial').
    All-zero column if the file is missing or empty.
    """
    short_name = category[4:] if category.startswith("poi_") else category
    count_col  = f"n_{short_name}"

    poi_path = poi_category_path(city, category)
    if not poi_path.exists():
        print(f"  [{city}] {category} POI file not found — {count_col} will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"], count_col: 0})

    poi = gpd.read_file(poi_path)
    if len(poi) == 0:
        print(f"  [{city}] {category} POI file is empty — {count_col} will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"], count_col: 0})

    poi = poi.to_crs(grid.crs)
    poi = poi[poi.geometry.geom_type.isin(["Point", "MultiPoint", "Polygon", "MultiPolygon"])].copy()

    polygon_mask = poi.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    poi.loc[polygon_mask, "geometry"] = poi.loc[polygon_mask, "geometry"].representative_point()

    if len(poi) == 0:
        return pd.DataFrame({"spatial_id": grid["spatial_id"], count_col: 0})

    joined = gpd.sjoin(poi, grid[["spatial_id", "geometry"]], how="inner", predicate="within")
    counts = joined.groupby("spatial_id").size().reset_index(name=count_col)

    out = grid[["spatial_id"]].merge(counts, on="spatial_id", how="left")
    out[count_col] = out[count_col].fillna(0).astype(int)

    if MAKE_PLOTS:
        plotted = grid.merge(out, on="spatial_id", how="left")
        fig, ax = plt.subplots(figsize=(8, 8))
        plotted.plot(column=count_col, ax=ax, legend=True,
                     edgecolor="black", linewidth=0.1, cmap="OrRd")
        if HAS_CONTEXTILY:
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=grid.crs)
            except Exception:
                pass
        ax.set_title(f"{city} — {short_name} POI count by tile")
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    print(f"  [{city}] {short_name} POIs — total: {out[count_col].sum()}")
    return out[["spatial_id", count_col]]


print("compute_poi_count() defined.")

### 4.6 Fossil-Fuel Power Plant Counts

In [ ]:
def compute_power_plants(city: str, grid: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Count fossil-fuel (coal/oil/gas) power plants per tile from the global
    power plant database. All-zero column if the database is missing.
    """
    if not POWER_PLANT_PATH.exists():
        print(f"  [{city}] Global power plant database not found — n_power_plant will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"], "n_power_plant": 0})

    plants = gpd.read_file(POWER_PLANT_PATH)
    if "primary_fuel" not in plants.columns:
        print(f"  [{city}] 'primary_fuel' column missing in power plant DB — n_power_plant will be 0.")
        return pd.DataFrame({"spatial_id": grid["spatial_id"], "n_power_plant": 0})

    plants = plants[plants["primary_fuel"].str.lower().isin(FOSSIL_FUEL_TYPES)].copy()
    plants = plants.to_crs(grid.crs)

    if len(plants) == 0:
        return pd.DataFrame({"spatial_id": grid["spatial_id"], "n_power_plant": 0})

    joined = gpd.sjoin(plants, grid[["spatial_id", "geometry"]], how="inner", predicate="within")
    counts = joined.groupby("spatial_id").size().reset_index(name="n_power_plant")

    out = grid[["spatial_id"]].merge(counts, on="spatial_id", how="left")
    out["n_power_plant"] = out["n_power_plant"].fillna(0).astype(int)

    if MAKE_PLOTS and out["n_power_plant"].sum() > 0:
        plotted = grid.merge(out, on="spatial_id", how="left")
        fig, ax = plt.subplots(figsize=(8, 8))
        plotted.plot(column="n_power_plant", ax=ax, legend=True,
                     edgecolor="black", linewidth=0.1, cmap="OrRd")
        if HAS_CONTEXTILY:
            try:
                cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, crs=grid.crs)
            except Exception:
                pass
        ax.set_title(f"{city} — Fossil fuel power plant count by tile")
        ax.set_axis_off()
        plt.show()
        plt.close(fig)

    print(f"  [{city}] Fossil-fuel power plants — total: {out['n_power_plant'].sum()}")
    return out[["spatial_id", "n_power_plant"]]


print("compute_power_plants() defined.")

## 5 · Build Static Feature Table — All Cities

For each city: load the grid once, run every feature block, merge everything
on `spatial_id`, fill count/length columns with 0, and save both a GeoPackage
(with geometry, for QC/mapping) and a lightweight CSV (no geometry, for the
`01c` merge). Cities with an existing output are skipped if `SKIP_EXISTING`.

In [ ]:
city_static_panels: dict = {}

for city in CITIES:
    out_gpkg = output_gpkg_path(city)
    if SKIP_EXISTING and out_gpkg.exists():
        print(f"\n⏭️  {city}: output already exists — skipping ({out_gpkg.name}).")
        city_static_panels[city] = gpd.read_file(out_gpkg)
        continue

    print("\n" + "=" * 70)
    print(f"  {city.upper()}")
    print("=" * 70)

    grid = gpd.read_file(grid_path(city))
    if "spatial_id" not in grid.columns:
        print(f"  ✗ 'spatial_id' column missing in grid — skipping {city}.")
        continue

    projected_crs = utm_crs_for(city, fallback_gdf=grid)

    final = grid[["spatial_id", "geometry"]].copy()

    # ── Population ────────────────────────────────────────────────────────
    final = final.merge(compute_population(city, grid), on="spatial_id", how="left")

    # ── Building height ───────────────────────────────────────────────────
    final = final.merge(compute_building_height(city, grid), on="spatial_id", how="left")

    # ── Roads ─────────────────────────────────────────────────────────────
    final = final.merge(compute_road_length(city, grid, projected_crs), on="spatial_id", how="left")

    # ── Public transport ──────────────────────────────────────────────────
    final = final.merge(compute_public_transport(city, grid), on="spatial_id", how="left")

    # ── POI categories ────────────────────────────────────────────────────
    for cat in POI_CATEGORIES:
        final = final.merge(compute_poi_count(city, grid, cat), on="spatial_id", how="left")

    # ── Power plants ──────────────────────────────────────────────────────
    final = final.merge(compute_power_plants(city, grid), on="spatial_id", how="left")

    # ── Fill count/length columns with 0 (never NaN) ──────────────────────
    fill_zero_cols = [
        c for c in final.columns
        if c.startswith("n_") or c.startswith("road_length_")
    ]
    final[fill_zero_cols] = final[fill_zero_cols].fillna(0)

    city_static_panels[city] = final
    print(f"\n  ✓ {city}: {len(final):,} tiles  |  columns: {list(final.columns)}")

print("\n✅ Static feature build complete.")

## 6 · Save Outputs

In [ ]:
for city, final in city_static_panels.items():
    out_gpkg = output_gpkg_path(city)
    out_csv  = output_csv_path(city)

    # GeoPackage (with geometry) — useful for QC/mapping
    final.to_file(out_gpkg, driver="GPKG")

    # CSV (no geometry) — lightweight join table consumed by 01c
    final.drop(columns="geometry").to_csv(out_csv, index=False)

    print(f"  {city}: saved")
    print(f"    {out_gpkg.relative_to(PROJECT_ROOT)}  ({out_gpkg.stat().st_size / 1e6:.2f} MB)")
    print(f"    {out_csv.relative_to(PROJECT_ROOT)}  ({out_csv.stat().st_size / 1e6:.2f} MB)")

print("\n✅ All outputs saved.")

## 7 · Summary

In [ ]:
print("=" * 70)
print("NOTEBOOK 01b5 SUMMARY")
print("=" * 70)

for city, final in city_static_panels.items():
    print(f"\n{city}")
    print(f"  Tiles        : {len(final):,}")
    print(f"  Columns      : {[c for c in final.columns if c != 'geometry']}")
    print(f"  tile_pop NaN : {final['tile_pop'].isna().sum() if 'tile_pop' in final.columns else 'n/a'}")
    if "building_height" in final.columns:
        print(f"  bh NaN       : {final['building_height'].isna().sum()}")
    print(f"  Output       : {output_gpkg_path(city).name}, {output_csv_path(city).name}")

print("\n→ Next step: 01c_feature_engineering.ipynb "
      "(merge {city}_static_features.csv into the daily panel, then TCI + lag)")